# L05 · Modern OPD from scratch

## Goal

**Estimated time:** 50 min · **Path:** fast, full

- connect student rollout to update
- distinguish full and sampled estimators
- block teacher gradients

### Current position: L04 → **L05** → L06

```text
Prompt/Data -> state source -> ... -> L05 -> ... -> fair evaluation
```

Alt text: The course map highlights L05 between its prerequisite and next lesson; every method remains connected to the same evaluation stage.

## Setup

In [1]:
LESSON_ID = "L05"
from pathlib import Path
import sys
import torch

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = Path.cwd().parents[1]
sys.path.insert(0, str(repo_root / "src"))

import opd_study
from opd_study.device import resolve_device
from opd_study.utils import seed_everything

seed_everything(42)
device_report = resolve_device("cpu")
print({"lesson": LESSON_ID, "opd_study": opd_study.__version__,
       "torch": torch.__version__, "device": device_report.selected,
       "profile": "toy", "network": "not required"})

{'lesson': 'L05', 'opd_study': '0.1.0.dev0', 'torch': '2.13.0', 'device': 'cpu', 'profile': 'toy', 'network': 'not required'}


## Steps

### 1/3 · 8–12 min

Sampling in modern sampled-token OPD is not differentiable. Rollout log-probabilities are snapshots; update-time student logits are recomputed so gradients use current parameters.

Figure alt: labels and numbers remain readable without color.

### Core mechanics

One update follows `collect → freeze trajectory → teacher score → recompute student logits → masked loss → optimizer step`. A discrete rollout token is not differentiated through. Rollout-time log-probabilities are audit/importance snapshots; gradients come from current-student log-probabilities recomputed on the sampled sequence.

Full reverse KL sums every vocabulary term, giving an exact, lower-variance token-state objective. Sampled reverse KL uses `(log p_student(y)-log p_teacher(y)).detach() * log p_student(y)` for `y ~ student`. It saves memory but introduces variance and baseline-design questions.

### Production implementation: why this design

`collect_student_trajectories` restores the student's previous train/eval mode and detaches rollout log-probability snapshots. Loss shifts one position and selects response targets only. Teacher/student logit shape mismatches fail before update.

Production code: [`rollout.py`](../../src/opd_study/algorithms/rollout.py), [`losses.py`](../../src/opd_study/algorithms/losses.py).

In [2]:
import inspect
from opd_study.algorithms import collect_student_trajectories, sampled_reverse_kl_loss

objects_to_show = (collect_student_trajectories, sampled_reverse_kl_loss,)
for object_to_show in objects_to_show:
    source_lines = inspect.getsource(object_to_show).splitlines()
    print(f"\n# {object_to_show.__module__}.{object_to_show.__qualname__}")
    print("\n".join(source_lines[:80]))
    if len(source_lines) > 80:
        print(f"... {len(source_lines) - 80} more lines; open the linked source file")


# opd_study.algorithms.rollout.collect_student_trajectories
def collect_student_trajectories(
    student: TinyCausalLM,
    prompts: Sequence[str],
    tokenizer: CharacterTokenizer,
    *,
    max_new_tokens: int = 64,
    min_new_tokens: int = 0,
    temperature: float = 1.0,
    generator: torch.Generator | None = None,
) -> TrajectoryBatch:
    """Sample responses and save detached rollout-time selected log-probabilities."""

    if not prompts:
        raise ValueError("prompts must not be empty")
    device = _module_device(student)
    sequences: list[Tensor] = []
    prompt_lengths: list[int] = []
    for prompt in prompts:
        prompt_tensor = torch.tensor(
            tokenizer.encode(prompt, bos=True), dtype=torch.long, device=device
        ).unsqueeze(0)
        if prompt_tensor.shape[1] >= student.config.max_sequence_length:
            raise ValueError("a prompt is too long for the student context window")
        generated = student.generate(
            prompt_ten

### Alternatives and trade-offs

Full estimators suit small vocabularies or available teacher logits. Sampled estimators suit large vocabularies/API log-probs but may need multiple samples, control variates, or clipping. Reusing old-policy rollouts requires importance correction and policy-version audits.

### 2/3 · Run and observe

Predict before running: which invariant should you inspect first in L05's output? Write one sentence, then run.

In [3]:
from opd_study.algorithms import (collect_student_trajectories,
    on_policy_distillation_loss, score_teacher)
from opd_study.data import CharacterTokenizer, generate_tiny_arithmetic
from opd_study.models import TinyCausalLM, TinyTransformerConfig

tokenizer = CharacterTokenizer(); splits = generate_tiny_arithmetic(train_rows=4, validation_rows=1, test_rows=1)
config = TinyTransformerConfig(vocab_size=tokenizer.vocab_size, number_of_layers=1,
    hidden_size=32, number_of_heads=4, feed_forward_size=64)
student, teacher = TinyCausalLM(config), TinyCausalLM(config)
trajectories = collect_student_trajectories(student, [row.prompt for row in splits.train[:2]],
    tokenizer, max_new_tokens=4, min_new_tokens=4, temperature=0.0)
signals = score_teacher(teacher, trajectories)
student_logits = student(trajectories.token_ids, trajectories.attention_mask)
output = on_policy_distillation_loss(student_logits, trajectories, signals)
print("shapes:", trajectories.token_ids.shape, student_logits.shape, output.token_loss.shape)
print("response tokens:", int(trajectories.response_mask.sum()), "loss:", float(output.loss.detach()))

shapes: torch.Size([2, 40]) torch.Size([2, 40, 81]) torch.Size([2, 40])
response tokens: 8 loss: 0.011338572017848492


In [4]:
optimizer = torch.optim.AdamW(student.parameters(), lr=1e-3)
optimizer.zero_grad(); output.loss.backward(); optimizer.step()
print("teacher has gradients:", any(parameter.grad is not None for parameter in teacher.parameters()))
print("rollout snapshot detached:", not trajectories.student_logprobs.requires_grad)

teacher has gradients: False
rollout snapshot detached: True


## Checks

In [5]:
assert not trajectories.response_mask[:, :trajectories.prompt_lengths.min()].any()
assert output.loss.requires_grad
assert not any(parameter.grad is not None for parameter in teacher.parameters())
print("check passed: sample -> detached state -> teacher no_grad -> recomputed student update")

check passed: sample -> detached state -> teacher no_grad -> recomputed student update


**Exercise (10 min):** remove `.detach()` from a copied sampled-RKL advantage, derive the extra gradient term, and compare gradients without editing production code.

<details><summary>Check</summary>Without detach, the advantage's student log-probability also differentiates, changing the intended score-function estimator.</details>

## My recurring mistakes

### M1 — Retaining the rollout graph through update

- Wrong: expect gradient through a discrete sampled token.
- Why: token selection is not differentiable.
- Fix: detach trajectories and recompute current logits.
- Related check: `test_rollout_snapshots_are_detached_and_mode_is_restored`

### M2 — Averaging KL over prompts and padding

- Wrong: directly mean the `[B,T]` loss.
- Why: prompt length and padding distort the budget.
- Fix: apply the shifted response mask with `masked_mean`.
- Related check: `test_sft_counts_only_response_targets`

## 60-second summary

1. connect student rollout to update
2. distinguish full and sampled estimators
3. block teacher gradients

## Next Steps

Before the next notebook, rerun the assertions and record one prediction you revised.

### Sources

- [`gkd`](https://arxiv.org/abs/2306.13649v3) · `2306.13649v3` · license `CC-BY-4.0` · [audited manifest](../../docs/sources.yml)
- [`vopd`](https://arxiv.org/abs/2605.07865v1) · `2605.07865v1` · license `CC-BY-4.0` · [audited manifest](../../docs/sources.yml)